![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 14 -- Lab 2: Word Embeddings -- From Words to Vectors

A digital library wants to build a smart search system. Instead of matching exact keywords, it should understand that "car" and "automobile" mean the same thing. In this lab you will use pre-trained word vectors to explore how AI represents the meaning of words as numbers.

| Part | Goal |
|---|---|
| Part 1 | Load word vectors and look at them |
| Part 2 | Find similar words |
| Part 3 | Word math: king - man + woman = queen |
| Part 4 | Build a smart search engine |

**All the library code is provided for you.** Your job is to explore, observe patterns, and answer questions.

---
## Setup

Run the cell below to install and import everything we need.

In [ ]:
!pip install gensim -q

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams['figure.dpi'] = 120

---
## Part 1: Load Word Vectors

We will use **GloVe** vectors -- pre-trained word embeddings where each word is represented as a list of 50 numbers. These were trained on billions of words from Wikipedia.

**Run the cell below** (it may take 1-2 minutes to download).

In [ ]:
import gensim.downloader as api

print("Downloading GloVe vectors (this takes 1-2 minutes)...")
glove = api.load("glove-wiki-gigaword-50")
print(f"Done! Loaded {len(glove):,} words, each represented by {glove.vector_size} numbers.")

### Exploring a Word Vector (GIVEN)

Let's look at what a word vector actually looks like. Run the cell below.

In [ ]:
king_vector = glove["king"]

print(f"The word 'king' is represented by {len(king_vector)} numbers:")
print(f"First 10 values: {king_vector[:10].round(3)}")
print()
print("Each number is one 'dimension' of meaning.")
print("No single number has a clear meaning on its own,")
print("but together they capture what the word means.")

### Task 1: Look at More Vectors

Print the first 10 values of the vectors for `"cat"`, `"dog"`, and `"car"`. You can access them with `glove["word"]`.

Do "cat" and "dog" look more similar to each other than to "car"? (It's hard to tell just by looking at numbers -- that's why we use math to measure similarity.)

In [ ]:
# Your code here

---
## Part 2: Finding Similar Words

**Cosine similarity** measures how close two word vectors are. It gives a score from -1 to 1:
- **1.0** = identical meaning
- **0.0** = completely unrelated
- **-1.0** = opposite meaning

### Task 2: Measure Similarity Between Word Pairs

The function `glove.similarity(word1, word2)` computes cosine similarity. Use it to fill in the results below.

Compute and print the similarity for these pairs:
- ("cat", "dog")
- ("cat", "car")
- ("king", "queen")
- ("king", "banana")
- ("happy", "sad")
- ("happy", "joyful")

Which pair is most similar? Which is least? Does it match what you expect?

In [ ]:
pairs = [
    ("cat", "dog"),
    ("cat", "car"),
    ("king", "queen"),
    ("king", "banana"),
    ("happy", "sad"),
    ("happy", "joyful"),
]

for w1, w2 in pairs:
    # Your code here: compute similarity and print it
    pass

### Task 3: Find the Nearest Neighbors

The function `glove.most_similar(word, topn=10)` finds the 10 words closest to the given word.

Find the 10 most similar words to:
1. `"computer"`
2. `"football"`
3. `"king"`

Do the results make sense? Do similar words cluster by category?

In [ ]:
# Your code here

### Task 4: Explore on Your Own

Pick 3 words that interest you and find their nearest neighbors. Try words from different categories -- a country, a sport, a feeling, a food, etc.

Write down anything surprising you find.

In [ ]:
# Your code here

---
## Part 3: Word Math (Analogies)

The most famous property of word embeddings: **you can do math with words!**

The idea: if "king" is to "man" as "queen" is to "woman", then:

**king - man + woman = queen**

This works because the vector from "man" to "king" captures the concept of "royalty", and adding that same direction to "woman" lands near "queen".

### The Classic Example (GIVEN)

Run the cell below to see word math in action.

In [ ]:
result = glove.most_similar(positive=["woman", "king"], negative=["man"], topn=5)

print("king - man + woman = ?")
print()
for word, score in result:
    print(f"  {word:15s}  (score: {score:.4f})")

### Task 5: Test More Analogies

Use `glove.most_similar(positive=[...], negative=[...], topn=5)` to solve each analogy. Print the top 5 results.

1. **paris - france + italy = ?** (expect: rome)
2. **japan - tokyo + france = ?** (expect: paris)
3. **dog - puppy + cat = ?** (expect: kitten)
4. **slow - slower + fast = ?** (expect: faster)

How many get the expected answer in the top 5?

In [ ]:
# Analogy 1: paris - france + italy = ?
# Your code here


In [ ]:
# Analogy 2: japan - tokyo + france = ?
# Your code here


In [ ]:
# Analogy 3: dog - puppy + cat = ?
# Your code here


In [ ]:
# Analogy 4: slow - slower + fast = ?
# Your code here


### Task 6: Invent Your Own Analogies

Come up with **3 analogies** of your own and test them. Some ideas:
- Country to capital: germany - berlin + egypt = ?
- Gender pairs: brother - man + woman = ?
- Comparative forms: big - bigger + small = ?

For each one, print whether the expected answer appears in the top 5.

In [ ]:
# Your code here

---
## Part 4: Build a Smart Search Engine

Now let's put word vectors to practical use. We will build a search engine that understands **meaning**, not just exact keywords.

**The idea:** represent each document as the average of its word vectors. To search, average the query's word vectors and find the most similar documents.

### Our Document Collection (GIVEN)

In [ ]:
documents = [
    "The cat sat on the warm windowsill watching birds outside",
    "Python is a popular programming language for data science",
    "The stock market crashed due to economic uncertainty",
    "Dogs are loyal companions and great family pets",
    "Machine learning algorithms can detect patterns in data",
    "The recipe calls for fresh tomatoes and basil leaves",
    "Electric vehicles are becoming more affordable each year",
    "The basketball team won the championship game last night",
    "Climate change is causing rising sea levels worldwide",
    "The new smartphone has an impressive camera system",
]

for i, doc in enumerate(documents):
    print(f"  [{i}] {doc}")

### Helper Functions (GIVEN)

These two functions do the heavy lifting. **Read the comments** to understand what they do, then run the cell.

In [ ]:
def document_vector(text, model):
    """Represent a text as the average of its word vectors."""
    words = text.lower().split()
    vectors = []
    for word in words:
        if word in model:
            vectors.append(model[word])
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


def search(query, documents, model, top_k=3):
    """Find the most relevant documents for a query."""
    query_vec = document_vector(query, model).reshape(1, -1)
    doc_vecs = np.array([document_vector(doc, model) for doc in documents])
    similarities = cosine_similarity(query_vec, doc_vecs)[0]
    ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)
    return [(documents[i], score) for i, score in ranked[:top_k]]

### Task 7: Test the Search Engine

Run the cell below to try several queries. Look at the results -- does the search engine find relevant documents?

In [ ]:
queries = ["pets and animals", "technology and computers", "cooking food", "sports"]

for query in queries:
    print(f"Query: '{query}'")
    print("-" * 55)
    results = search(query, documents, glove, top_k=3)
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  {rank}. (score: {score:.3f}) {doc}")
    print()

### Task 8: Semantic Search vs Keyword Search

Try these queries that use **different words** than any document. Does the search engine still find the right document?

1. `"feline"` -- should match the cat document
2. `"automobile"` -- should match the electric vehicles document
3. `"cuisine"` -- should match the recipe document
4. `"money finance"` -- should match the stock market document

In [ ]:
semantic_queries = ["feline", "automobile", "cuisine", "money finance"]

for query in semantic_queries:
    print(f"Query: '{query}'")
    results = search(query, documents, glove, top_k=1)
    print(f"  Best match: {results[0][0]}")
    print(f"  Score: {results[0][1]:.3f}")
    print()

### Task 9: Try Your Own Queries

Come up with 3 creative queries and test them. Try queries that:
- Use synonyms of words in the documents
- Describe the topic without using any words from the document
- Are in a completely different style

In [ ]:
# Your code here

### Task 10: Discussion Questions

1. Why can the search engine find relevant documents even when the query uses completely different words?

2. Can you think of a query where this approach would fail? (Hint: what if the meaning depends on word order?)

3. How is this different from searching with Ctrl+F (exact keyword matching)?

*Write your answers here.*

---
## Wrap-Up

**Key takeaways:**

- Word embeddings represent words as lists of numbers (vectors) where **similar words have similar vectors**
- Cosine similarity measures how related two words are
- Word math works: king - man + woman = queen (vector arithmetic captures relationships)
- Averaging word vectors gives a simple way to represent whole sentences
- Semantic search finds relevant results even with different wording

**Next:** In Lab 3, you will visualize these 50D word vectors in 2D using PCA, t-SNE, and UMAP -- and see clusters of animals, countries, and verbs emerge from the data.